# Calvin Site 1 cutoff and QC review

Compare the measured cylinders at 0.50 and 0.93 mm on their common support.
Default cells read the saved local results. The six-seed panels hold geometry fixed across observation cutoffs.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess
import pandas as pd
from IPython.display import display, Image

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if
               (p / '.runtime/calvin-cutoff-qc-20261119/targets/preparation.json').exists())
study = project / '.runtime/calvin-cutoff-qc-20261119'
target_manifest = json.loads((study / 'targets/preparation.json').read_text())
for name, expected in target_manifest['artifacts_sha256'].items():
    assert sha256((study / 'targets' / name).read_bytes()).hexdigest() == expected
display(pd.read_csv(study / 'targets/site_support.csv'))

In [ ]:
comparison = study / 'comparison/scores'
all_scores = pd.read_csv(comparison / 'scores.csv')
held = all_scores.query('support == "common" and split == "Site1_hold"')
fields = ['cutoff_mm', 'qc_view', 'case', 'increments', 'spatial_score',
          'rld_nmae', 'mass_nmae', 'rld_joint_tv', 'mass_joint_tv', 'rld_depth_tv', 'mass_depth_tv']
display(held[fields].round(3))
display(Image(filename=str(study / 'depth_allocations.png'), width=1050))

## Rebuild and replay

Run the following cell with `REPLAY = True` to verify the recorded trial receipts and recompute the matched scores.
The comparison scripts skip existing trials bound to the frozen protocol. A missing receipt stops this notebook replay.
A fresh raw-workbook extraction needs an interpreter with openpyxl and a new output path.

In [ ]:
REPLAY = False
python310 = project / '.runtime/envs/py310-qualification/bin/python'
source = project / '.runtime/spatial-construction-20261119/baseline'
env = dict(os.environ, PYTHONPATH=str(source), PYTHONDONTWRITEBYTECODE='1',
           OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
if REPLAY:
    protocol = json.loads((study / 'comparison/protocol.json').read_text())
    for case in protocol['cases']:
        for seed in protocol['seeds']:
            receipt = study / 'comparison/trials' / case / f'{seed}.json'
            assert json.loads(receipt.read_text())['status'] == 'passed'
    for panel in ['fixed', 'gate']:
        subprocess.run([python310, study / 'rescore_cutoff.py', panel], cwd=source, env=env, check=True)
    subprocess.run([python310, study / 'score_cutoffs.py'], cwd=source, env=env, check=True)
print('Target rebuild:', 'python with openpyxl', study / 'prepare_cutoff.py', '--outdir <new-directory>')